# Lab: weighing the ice sheets with GRACE

```{note}
This lab accompanies {doc}`gravity`. The chapter described how inter-satellite ranging converts the separation history of two spacecraft into a time-varying gravity field, and how that field is the only glaciological observable that measures ice-mass change without a density assumption. Here you will fetch the processed data, extract ice-sheet time series, fit trends, and confront the three honesty problems that every GRACE practitioner carries: leakage, glacial isostatic adjustment, and the 2017–2018 gap between the two missions.
```

By the end of this lab you should be able to

- explain why Level-3 mascon solutions are the practical entry point for most scientists, and what was done upstream to produce them,
- open the JPL mascon netCDF, interpret its dimensions and scale factors, and apply the land mask correctly,
- convert a gridded water-equivalent anomaly to a regional mass time series in gigatonnes,
- fit a trend plus seasonal model by ordinary least squares and estimate the acceleration,
- subtract a GIA model estimate and show what fraction of the apparent Antarctic trend it explains,
- handle the GRACE–GRACE-FO gap honestly rather than pretending it is not there,
- close the mass budget three ways and state what agreement among the methods means.

The tools are `earthaccess`, `xarray`, `numpy`, and `matplotlib`. The optional comparison in Task 2 uses a second mascon center if you have downloaded both files.

## Where the data lives, and what "raw" means here

Every observing chapter in this part has started at the same question: where does the raw data live, and how far from the satellite record do you have to travel before the number you plot is defensible? For GRACE the answer is unusual enough to state plainly.

The true raw product, Level-1B, is a time series of the inter-satellite range and range-rate measured by the K-Band Ranging system at the micrometres-per-second level {cite}`tapley2004`. Turning that into a gravity field requires solving a large regularised inversion for spherical harmonic coefficients or for mass concentrations (mascons), removing modelled contributions from the atmosphere and ocean, and handling the aliasing that comes from sampling a continuously varying field once per orbital repeat. Only a handful of processing centres — JPL, CSR at the University of Texas, and GSFC at NASA Goddard — have the infrastructure and the accumulated expertise to do that reliably. For almost everyone else, the practical starting point is their Level-3 output: monthly gridded estimates of the equivalent water height (EWH) anomaly at roughly one-degree resolution.

That is not a limitation to apologise for; it is a division of labour. The authors of the Level-3 products have published their methods, their error budgets, and the scale factors that correct for spectral leakage between neighbouring basins, and you can read and evaluate those choices. What you gain by starting at Level-3 is the ability to do science on the mass signal rather than on the signal processing, which is the lab's objective.

**A note on data volume.** Gravity is the rare case in this part where the data is genuinely small. The GRACE and GRACE-FO mascon solutions together, covering roughly 2002–present, fit in a few hundred megabytes as a single netCDF. Compare that with the tens of gigabytes of a single ICESat-2 granule in the laser-altimetry chapter, or the multi-gigabyte SLC pairs of {doc}`insar`. The reason is the resolution: the gravity field is smooth at scales below a few hundred kilometres because shorter-wavelength anomalies decay exponentially with altitude, so the grid is coarse and the file is small. The coarseness is not a defect of the processing; it is a physical limit of the method.

In [ ]:
import numpy as np
import xarray as xr
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from pathlib import Path

# earthaccess handles NASA Earthdata authentication and the PO.DAAC download.
# Install with:  pip install earthaccess
import earthaccess

plt.rcParams.update({'figure.dpi': 120, 'font.size': 10})

## Downloading the JPL mascon solution

The JPL RL06v3 mascon solution lives on PO.DAAC (Physical Oceanography DAAC) and is freely accessible to anyone with a NASA Earthdata account. The `earthaccess` library handles the authentication and the download in a few lines. The short name for the combined GRACE and GRACE-FO mascon collection is given in the cell below; verify it has not changed since this notebook was written by searching PO.DAAC or by running `earthaccess.search_datasets(keyword='GRACE mascon JPL RL06')`.

The file downloaded here is a single netCDF that packages all monthly solutions from April 2002 through the present GRACE-FO record. Opening it in xarray gives you a `time` dimension (one entry per month), `lat` and `lon` dimensions at 0.5-degree resolution, and the variable `lwe_thickness` — liquid water equivalent thickness in centimetres. There is also a `scale_factor` grid, a land-ocean mask, and the GIA model that JPL used in its own processing (which you will not subtract automatically, but will examine explicitly).

In [ ]:
# verify: collection short name — check PO.DAAC if this has been updated
COLLECTION_SHORTNAME = 'TELLUS_GRAC-GRFO_MASCON_CRI_TIME_SERIES_RL06.3_V4'

LOCAL_DIR = Path('data/grace')
LOCAL_DIR.mkdir(parents=True, exist_ok=True)

# Authenticate with NASA Earthdata (prompts once; caches credentials).
# earthaccess.login(strategy='interactive')

# results = earthaccess.search_data(short_name=COLLECTION_SHORTNAME)
# earthaccess.download(results, LOCAL_DIR)

# For the purposes of this lab, point GRACE_FILE at the downloaded path:
GRACE_FILE = LOCAL_DIR / 'GRACE_GRACE-FO_RL06.3_JPL_mascons_CRI.nc'

print('Expected file location:', GRACE_FILE)
print('File exists:', GRACE_FILE.exists())

## Inspecting the file: dimensions, masks, and scale factors

Before extracting any time series it is worth opening the file and reading it as a document, not just as a data source. The variable names, their units, the flag values in the mask, and the shape of the scale-factor grid each carry information about what the processing centres chose to do and what they chose to warn you about.

The scale factors deserve particular attention. The mascon inversion applies regularisation that tends to smooth the solution, smearing mass from a small basin (say, the Amundsen Sea sector of West Antarctica) into neighbouring ocean or land pixels. The scale factors — one per grid cell — are empirically derived correction multipliers that attempt to reverse this leakage. Multiplying `lwe_thickness` by `scale_factor` before any regional integration is not optional; regions with sharp boundaries and high mass-change rates (Pine Island Glacier, the Greenland southeast coast) can be underestimated by 20–30 per cent without it.

The land-ocean mask tells you which cells to include when you want only the ice sheet and not the adjacent sea. GRACE cannot distinguish ice from land from ocean within its footprint, so the mask is your way of telling the integral what to sum over. Use it strictly: ocean-bordering ice-sheet margins are exactly the regions where leakage is worst, and including sea cells there would double-count the correction the scale factor is already applying.

In [ ]:
# ds = xr.open_dataset(GRACE_FILE)
# print(ds)
# print('\nVariables:')
# for v in ds.data_vars:
#     print(f'  {v}: {ds[v].dims}  units={ds[v].attrs.get("units", "?")}  '
#           f'shape={ds[v].shape}')

# The key variables we will use:
#   lwe_thickness  : monthly EWH anomaly, cm  (dims: time, lat, lon)
#   scale_factor   : leakage correction grid  (dims: lat, lon)
#   land_mask      : 1 = land/ice, 0 = ocean  (dims: lat, lon)
#   time_coverage_start, time_coverage_end: per-month strings

# Inspect the scale factor range over Antarctica and Greenland:
# ant_mask  = (ds.lat < -60) & (ds.land_mask == 1)
# grn_mask  = (ds.lat > 60) & (ds.lat < 85) & (ds.land_mask == 1)
# print('Antarctic scale factor range:',
#       float(ds.scale_factor.where(ant_mask).min().values),
#       'to',
#       float(ds.scale_factor.where(ant_mask).max().values))

# print('\nTime span:', ds.time.values[0], 'to', ds.time.values[-1])
# print('Number of monthly solutions:', ds.dims['time'])

print('Cell not executed at build — run locally after downloading the file.')

## Building ice-sheet mass time series

The conversion from a gridded EWH anomaly to a total mass anomaly in gigatonnes requires three steps. First, apply the scale factor. Second, mask to the region of interest using the land mask together with a latitude and longitude box that surrounds the ice sheet. Third, area-weight the sum: each grid cell covers a different physical area because a degree of longitude shrinks as the cosine of latitude, so summing without weighting would overcount the tropics and undercount the poles — the opposite of what you want when you are working at high latitudes.

The area of a grid cell at latitude $\phi$ with angular width $\Delta\phi$ and $\Delta\lambda$ is

$$ A = R_e^2 \, \Delta\lambda \, \cos\phi \, \Delta\phi, $$

where $R_e = 6.371 \times 10^6$ m and all angles are in radians. The EWH anomaly in metres times the cell area in square metres gives a volume of water in cubic metres; dividing by $10^{12}$ converts to a mass in gigatonnes (using water density $\rho_w = 1000$ kg m$^{-3}$ and $1$ Gt $= 10^{12}$ kg).

The resulting time series is the mass anomaly relative to a baseline mean, not a total mass. GRACE always measures changes, never absolute magnitudes; the baseline is typically the 2004–2009 mean of the mission record, chosen because the early and late years have more missing months.

In [ ]:
Re    = 6.371e6     # Earth radius, m
rho_w = 1000.0      # water density, kg/m^3
GT    = 1.0e12      # 1 Gt in kg


def ewh_to_Gt(ds, lat_min, lat_max, lon_min=None, lon_max=None,
              apply_scale=True, land_only=True):
    """Integrate scaled EWH over a lat/lon box to get mass anomaly in Gt.

    Parameters
    ----------
    ds        : xarray Dataset from the JPL mascon netCDF
    lat_min, lat_max : bounding latitudes (degrees)
    lon_min, lon_max : bounding longitudes (degrees); if None, full circle
    apply_scale : multiply by scale_factor before integrating
    land_only  : restrict to land_mask == 1 cells

    Returns
    -------
    time : 1-D array of datetime64
    mass : 1-D array of mass anomaly in Gt (same length as time)
    """
    # --- spatial selection ---
    region = ds.sel(lat=slice(lat_min, lat_max))
    if lon_min is not None and lon_max is not None:
        region = region.sel(lon=slice(lon_min, lon_max))

    ewh = region['lwe_thickness'] * 1e-2   # cm -> m
    if apply_scale:
        ewh = ewh * region['scale_factor']

    # --- area weights ---
    lat_rad  = np.deg2rad(region.lat.values)
    dlat_rad = np.abs(np.deg2rad(region.lat.values[1] - region.lat.values[0]))
    dlon_rad = np.abs(np.deg2rad(region.lon.values[1] - region.lon.values[0]))
    area     = Re**2 * np.cos(lat_rad) * dlat_rad * dlon_rad   # m^2, shape (nlat,)
    area_2d  = xr.DataArray(area[:, None] * np.ones(region.dims['lon']),
                             dims=['lat', 'lon'])

    # --- land mask ---
    if land_only:
        mask = region['land_mask'].values.astype(bool)
        area_2d = area_2d.where(xr.DataArray(mask, dims=['lat', 'lon']))

    # --- integrate (volume of water -> mass in Gt) ---
    mass_Gt = (ewh * area_2d).sum(dim=['lat', 'lon']) * rho_w / GT
    return region.time.values, mass_Gt.values


# --- Greenland and Antarctic time series ---
# t_grn,  M_grn  = ewh_to_Gt(ds, lat_min=60,  lat_max=85)
# t_ant,  M_ant  = ewh_to_Gt(ds, lat_min=-90, lat_max=-60)

# --- remove the 2004-2009 baseline mean ---
# def remove_baseline(t, M, y0=2004, y1=2009):
#     base = (t.astype('datetime64[Y]').astype(int) + 1970 >= y0) & \
#            (t.astype('datetime64[Y]').astype(int) + 1970 <= y1)
#     return M - M[base].mean()

# M_grn = remove_baseline(t_grn, M_grn)
# M_ant = remove_baseline(t_ant, M_ant)

print('Function defined. Run the commented lines after loading the dataset.')

## Fitting trend, annual cycle, and acceleration

The mass time series you just built contains at least three separable signals riding on top of each other. There is a long-term trend — what we care about most. There is a strong annual cycle driven by the seasonal accumulation and melt cycle; over Greenland this can reach 300–500 Gt peak-to-peak, which is several times larger than the annual trend. And there may be an acceleration: the trend is not stationary, and the acceleration term is the quantity that feeds directly into sea-level projections {cite}`velicogna2006`.

The standard approach fits the following model by ordinary least squares to the mass anomaly $M(t)$:

$$
M(t) = a + b\,(t - t_0) + c\,(t - t_0)^2
       + A_1 \sin(2\pi t) + B_1 \cos(2\pi t)
       + A_2 \sin(4\pi t) + B_2 \cos(4\pi t)
       + \varepsilon(t),
$$

where $t$ is in decimal years and $t_0$ is the mid-point of the record. The coefficient $b$ is the mean trend in Gt yr$^{-1}$, and $2c$ is the acceleration in Gt yr$^{-2}$. The annual and semiannual terms ($A_1, B_1, A_2, B_2$) absorb the seasonal variability so that the trend estimate is not contaminated by whatever fraction of the record length is an integer multiple of a year.

Uncertainties on $b$ and $c$ require some care. The residual $\varepsilon(t)$ is not white noise; GRACE mass anomalies show correlated noise at the month-to-month level, which inflates the formal least-squares uncertainties. A conservative approach scales the formal standard errors by the square root of the estimated autocorrelation length, or uses a block-bootstrap. The uncertainties below will be the formal OLS values; treat them as lower bounds.

In [ ]:
def decimal_year(t_dt64):
    """Convert numpy datetime64 array to decimal years."""
    yr  = t_dt64.astype('datetime64[Y]').astype(float) + 1970.0
    doy = (t_dt64 - t_dt64.astype('datetime64[Y]')).astype('timedelta64[D]').astype(float)
    return yr + doy / 365.25


def fit_trend(t_yr, M, return_model=True):
    """Fit trend + annual + semiannual to mass time series by OLS.

    Returns a dict with keys: a, trend_Gt_yr, accel_Gt_yr2,
    A1, B1, A2, B2, residuals, and optionally M_fit.
    """
    t0  = 0.5 * (t_yr.min() + t_yr.max())
    tau = t_yr - t0
    pi2 = 2.0 * np.pi

    # design matrix: [1, tau, tau^2, sin(1/yr), cos(1/yr), sin(2/yr), cos(2/yr)]
    G = np.column_stack([
        np.ones_like(tau),
        tau,
        tau**2,
        np.sin(pi2 * t_yr),
        np.cos(pi2 * t_yr),
        np.sin(2.0 * pi2 * t_yr),
        np.cos(2.0 * pi2 * t_yr),
    ])

    m, _res, _rank, _sv = np.linalg.lstsq(G, M, rcond=None)
    result = dict(
        a            = m[0],
        trend_Gt_yr  = m[1],
        accel_Gt_yr2 = 2.0 * m[2],   # derivative of the quadratic term
        A1=m[3], B1=m[4], A2=m[5], B2=m[6],
        residuals    = M - G @ m,
    )
    if return_model:
        result['M_fit'] = G @ m
    return result


# Apply to both records:
# t_yr_grn = decimal_year(t_grn)
# t_yr_ant = decimal_year(t_ant)
# fit_grn  = fit_trend(t_yr_grn, M_grn)
# fit_ant  = fit_trend(t_yr_ant, M_ant)

# for name, fit in [('Greenland', fit_grn), ('Antarctica', fit_ant)]:
#     print(f'{name}: trend = {fit["trend_Gt_yr"]:+.1f} Gt/yr   '
#           f'acceleration = {fit["accel_Gt_yr2"]:+.2f} Gt/yr^2')

print('Fit functions defined.')

## The mass-loss curves

Plot the deseasonalised mass anomaly — that is, $M(t)$ minus the fitted annual and semiannual terms — together with the full fitted model for both ice sheets. The canonical result, first clearly visible in the early GRACE record and reproduced countless times since, is that Greenland and Antarctica are both losing mass, and the rate is not constant. Greenland loss accelerated sharply through the 2000s before levelling in the early 2010s and then accelerating again. The Antarctic signal is smaller in trend magnitude but carries a larger GIA uncertainty, which the next section will address.

The acceleration term $2c$ is the quantity that appears in sea-level projections. Even a small positive value, compounded over a century, produces far more sea-level rise than a constant trend would. But the uncertainty in $c$ is large enough that the sign of the acceleration is not always statistically secure for any individual decade-long sub-period; it is in the full record that the signal emerges.

In [ ]:
# fig, axes = plt.subplots(2, 1, figsize=(10, 7), sharex=True)

# datasets = [
#     ('Greenland',  t_grn,  M_grn,  fit_grn,  'steelblue'),
#     ('Antarctica', t_ant,  M_ant,  fit_ant,  'indianred'),
# ]

# for ax, (name, t, M, fit, color) in zip(axes, datasets):
#     t_yr = decimal_year(t)
#     seasonal = (fit['A1'] * np.sin(2*np.pi*t_yr) +
#                 fit['B1'] * np.cos(2*np.pi*t_yr) +
#                 fit['A2'] * np.sin(4*np.pi*t_yr) +
#                 fit['B2'] * np.cos(4*np.pi*t_yr))
#     M_deseas = M - seasonal

#     ax.plot(t, M_deseas, color=color, lw=0.8, alpha=0.7, label='deseasonalised')
#     ax.plot(t, fit['M_fit'] - seasonal, color='k', lw=2, label='trend + accel')

#     ax.set_ylabel('Mass anomaly (Gt)')
#     ax.set_title(
#         f'{name}:  {fit["trend_Gt_yr"]:+.0f} Gt/yr   '
#         f'acceleration {fit["accel_Gt_yr2"]:+.1f} Gt/yr²'
#     )
#     ax.legend(loc='lower left')
#     ax.axhline(0, color='gray', lw=0.5)

#     # shade the GRACE–GRACE-FO gap
#     ax.axvspan(np.datetime64('2017-07'), np.datetime64('2018-06'),
#                alpha=0.15, color='gold', label='mission gap')

# axes[1].xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
# axes[1].set_xlabel('Year')
# plt.tight_layout()
# plt.show()

print('Plot cell — run after loading data and fitting.')

## The GIA correction and what it does to Antarctica

Glacial isostatic adjustment (GIA) is the slow elastic and viscous rebound of the solid Earth under regions that were glaciated during the Last Glacial Maximum and are still responding to their deglaciation. Rock moving upward beneath an ice sheet adds mass to the volume GRACE measures even though the ice itself has not changed. Over Greenland the GIA signal is modest, a few gigatonnes per year, because the rock there rebounds relatively quickly due to its thin lithosphere and warm upper mantle. Over Antarctica the correction is large and uncertain: estimates from different GIA models disagree by 50–100 Gt yr$^{-1}$, which is comparable to the total Antarctic ice-loss trend in early GRACE periods.

The connection to the chapter on ice-sheet dynamics in {doc}`../cryosphere/ice-sheets` is direct. The GIA models are built on reconstructions of past ice-sheet extent and thickness, on models of the solid Earth's rheology, and on constraints from GPS observations of present-day bedrock uplift rates. Where those reconstructions are uncertain — and they are most uncertain beneath the East Antarctic Ice Sheet, where the bedrock is largely unsurveyed — the GIA correction carries that uncertainty into every GRACE trend estimate.

In the cell below you will apply a simple spatially uniform GIA correction to the Antarctic time series and show what fraction of the raw trend it removes. A real application would use a published model (ICE-6G, W12, or Whitehouse et al.) on the same grid as the mascon solution and subtract it month by month.

In [ ]:
# Published GIA model estimates for Antarctica range from about +50 to +150 Gt/yr
# (positive = rock gaining mass = must be subtracted from GRACE to isolate ice).
# We use a midrange estimate here; see [TODO-CITE: Whitehouse GIA Antarctic review]
# for a synthesis of published values.

GIA_ANT_Gt_yr = 100.0   # Gt/yr; replace with spatially integrated model value

# The GIA contribution to the time series is a linear ramp:
# GIA_signal(t) = GIA_ANT_Gt_yr * (t - t0)
# Subtracting it shifts the trend by GIA_ANT_Gt_yr.

# ant_raw_trend  = fit_ant['trend_Gt_yr']
# ant_ice_trend  = ant_raw_trend - GIA_ANT_Gt_yr   # GIA is additive, so subtract

# print(f'Antarctic raw GRACE trend:    {ant_raw_trend:+.1f} Gt/yr')
# print(f'GIA correction applied:        {-GIA_ANT_Gt_yr:+.1f} Gt/yr')
# print(f'GIA-corrected ice-mass trend: {ant_ice_trend:+.1f} Gt/yr')
# print(f'GIA fraction of raw trend:    '
#       f'{100 * GIA_ANT_Gt_yr / abs(ant_raw_trend):.0f}%')

# The large fraction demonstrates why GIA is the dominant uncertainty
# in Antarctic GRACE estimates — and why independent constraints on
# bedrock uplift from continuous GPS networks matter so much.

print('GIA cell — run after fitting the Antarctic time series.')

## The 2017–2018 gap and how to treat it honestly

GRACE's last science data were collected in October 2017. GRACE-FO was launched in May 2018 and began producing science data in June 2018. The gap between the two missions is about eleven months, and it sits in the middle of a period of rapid Greenland mass loss and notable Antarctic variability. How you treat it determines whether your long-term trend estimate is defensible or not.

Three approaches are in common use, none of them entirely satisfying.

The first is to leave the gap as missing data and fit your trend model to the two segments separately, then report the pre-gap and post-gap trends independently. This is the most honest approach and the one that avoids any assumption about what happened during the eleven months of silence.

The second is to interpolate across the gap using the fitted seasonal model. Because you have several years of data on either side, you can predict the seasonal cycle during the gap with reasonable confidence; what you cannot predict is any anomalous mass change that happened to occur in those months. The interpolated points carry a large uncertainty and should be flagged rather than treated as data.

The third is to bridge the gap using altimetry from ICESat-2, which was operating continuously through this period. Converting a volume change to a mass change requires a density assumption, which reintroduces the limitation that GRACE exists to avoid, but if the density field is well constrained the combined record is better than either alone.

In the shaded region of the mass-loss plots above, you used the first approach (leaving the gap visible). The task below asks you to implement the second and show explicitly what uncertainty the interpolation carries.

In [ ]:
# Define the gap boundaries.
GAP_START = np.datetime64('2017-11')
GAP_END   = np.datetime64('2018-06')

# --- Gap-bridging by seasonal interpolation ---
#
# 1. Generate synthetic monthly times covering the gap.
# gap_times = np.arange(GAP_START, GAP_END, dtype='datetime64[M]')
# gap_t_yr  = decimal_year(gap_times.astype('datetime64[ns]'))
#
# 2. Evaluate only the fitted seasonal model (no trend) at those times.
#    The trend at the gap mid-point is unknown; flag with NaN.
# t0  = 0.5 * (t_yr_grn.min() + t_yr_grn.max())
# seasonal_gap = (
#     fit_grn['A1'] * np.sin(2*np.pi*gap_t_yr) +
#     fit_grn['B1'] * np.cos(2*np.pi*gap_t_yr) +
#     fit_grn['A2'] * np.sin(4*np.pi*gap_t_yr) +
#     fit_grn['B2'] * np.cos(4*np.pi*gap_t_yr)
# )
# trend_at_gap = fit_grn['a'] + fit_grn['trend_Gt_yr'] * (gap_t_yr - t0)
# M_gap_interp = trend_at_gap + seasonal_gap
#
# 3. Estimate the uncertainty as the RMS residual of the fit,
#    scaled by sqrt(n_gap_months) to account for the autocorrelation.
# rms_resid = np.sqrt(np.mean(fit_grn['residuals']**2))
# sigma_gap  = rms_resid * np.sqrt(len(gap_times))
# print(f'Gap-bridge uncertainty (1-sigma): {sigma_gap:.0f} Gt')

print('Gap handling cell — run after fitting.')

## Tasks

**Task 1: Basin-scale versus whole-sheet trends.**

The whole-Antarctic trend masks large regional contrasts. The East Antarctic Ice Sheet (EAIS, roughly $-90^\circ$ to $45^\circ$ E and $45^\circ$ E to $180^\circ$ E) has been close to mass balance or slightly gaining; the West Antarctic Ice Sheet (WAIS, roughly $-140^\circ$ to $-60^\circ$ E) has been losing mass rapidly, driven by the glaciers draining the Amundsen Sea sector. Adapt `ewh_to_Gt` to extract EAIS and WAIS separately using bounding boxes, fit trends to each, and compare the basin trends with the continental sum. Is the WAIS loss rate consistent with what the input-output method finds for Pine Island and Thwaites? (You will need the discharge numbers from {doc}`../observing/insar`.)

**Task 2: Comparing mascon centres.**

If you have downloaded a second mascon solution — CSR RL06 mascons are available from the same PO.DAAC collection search — repeat the Greenland and Antarctic time series extraction and trend fits and compare the two. Report the difference in trend magnitude (Gt yr$^{-1}$) and in acceleration (Gt yr$^{-2}$). The differences reflect different processing choices — regularisation strength, GIA model used internally, scale-factor derivation — and give you a sense of the inter-centre spread that is a lower bound on the true systematic uncertainty. Note that direct numerical comparison is only meaningful after both files have been processed with the same baseline-removal convention.

For both tasks, print a clean summary table before producing any plots.

In [ ]:
# Task 1 basin boxes (approximate; see [TODO-CITE: Shepherd IMBIE Antarctica 2018]
# for the official IMBIE basin definitions)
BASINS = {
    'EAIS': dict(lat_min=-90, lat_max=-60, lon_min=0,    lon_max=180),
    'WAIS': dict(lat_min=-90, lat_max=-60, lon_min=-140, lon_max=-60),
    'AIS_full': dict(lat_min=-90, lat_max=-60),
}

# for basin, kw in BASINS.items():
#     t_b, M_b = ewh_to_Gt(ds, **kw)
#     t_yr_b   = decimal_year(t_b)
#     fit_b    = fit_trend(t_yr_b, M_b)
#     print(f'{basin:12s}: {fit_b["trend_Gt_yr"]:+.1f} Gt/yr  '
#           f'accel {fit_b["accel_Gt_yr2"]:+.2f} Gt/yr^2')

# Task 2: path to the CSR mascon file if available
CSR_FILE = Path('data/grace/CSR_GRACE_GRACE-FO_RL06_Mascons_all-corrections_v02.nc')
# if CSR_FILE.exists():
#     ds_csr = xr.open_dataset(CSR_FILE)
#     # CSR variable names differ slightly — check ds_csr.data_vars
#     # and adapt ewh_to_Gt as needed before comparing.
#     pass

print('Tasks cell — run locally with data files present.')

## Synthesis: closing the mass budget three ways

The three independent methods for measuring ice-sheet mass change are altimetry, input-output, and gravimetry. Each measures something different and carries a different combination of uncertainties.

**Altimetry** measures volume change, from which mass change follows only after assuming or modelling the density of the material whose surface changed. Where the surface lowering is dominated by dynamic thinning, the ice density of about 917 kg m$^{-3}$ applies. Where it reflects a change in the firn column, the relevant density can be anywhere from 400 to 917 kg m$^{-3}$ and must come from a firn densification model. Laser altimetry from ICESat and ICESat-2 {cite}`markus2017` gives this with high along-track precision; radar altimetry from CryoSat-2 {cite}`wingham2006` provides wider spatial coverage but lower vertical precision and a penetration correction for the radar signal in dry firn.

**Input-output** computes the residual between the snow accumulation integrated over the ice-sheet interior (from reanalysis or regional climate models) and the solid ice discharge across the grounding line (from ice velocity times thickness) {cite}`mouginot2019`. It is sensitive to accumulation variability, to errors in the bed topography used to compute cross-sectional area, and to the location and completeness of the grounding-line velocity map. Its strength is that it attributes mass loss to a process: discharge exceeding accumulation is unambiguously dynamic.

**Gravimetry** measures mass change directly, as you have seen, but requires the GIA correction and is blind to the distribution of that mass change within the footprint of a single mascon.

The IMBIE consortium has compared all three methods for both ice sheets over the full GRACE and post-GRACE record. The agreement is not perfect — the methods disagree by 10–30 per cent for individual basins and individual decades — but the sign and approximate magnitude of mass loss are consistent across all three. That agreement is the strongest statement observational glaciology can make to the ice-flow models of {doc}`../ice_flow/mass-balance`. When three instruments built on different physical principles, processed by independent groups, and carrying different systematic errors all report that the Amundsen Sea glaciers are losing mass at an accelerating rate, the probability that it is an artefact of any single method becomes vanishingly small.

The table below summarises representative values. Fill in the blanks from your own fits and from the literature.

In [ ]:
import pandas as pd

# Indicative synthesis values; update from your fits and
# [TODO-CITE: IMBIE team Greenland 2020] and [TODO-CITE: IMBIE team Antarctica 2018]
budget_data = {
    'Method': ['Gravimetry (GRACE/FO)', 'Altimetry (ICESat-2/CryoSat)', 'Input-output'],
    'Greenland trend (Gt/yr)': ['YOUR FIT', '-280 ± 45', '-275 ± 30'],
    'Antarctica trend (Gt/yr)': ['YOUR FIT (GIA-corrected)', '-155 ± 40', '-148 ± 55'],
    'Dominant uncertainty': ['GIA model', 'Firn density', 'Accumulation model'],
}

df = pd.DataFrame(budget_data)
print(df.to_string(index=False))
print()
print('Replace "YOUR FIT" with results from fit_grn and fit_ant after running'
      ' the data cells above.')

## Closing reflection

Gravity occupies a peculiar position among the observing methods in this part. Its raw observable — the changing distance between two refrigerator-sized satellites 220 km apart, measured to the width of a red blood cell — is so far from the scientific quantity of interest that you cannot trace the processing yourself without specialised code and computing resources. Yet the quantity it delivers — mass in gigatonnes — is arguably the most direct measurement glaciology possesses. No density assumption, no velocity-thickness product, no atmospheric correction for the target variable itself.

The honesty required of a GRACE user is different from the honesty required of an InSAR or altimetry user. It is not about opening every raw file and rerunning every pipeline; it is about being explicit on three fronts: the GIA correction and its range of published values, the leakage correction and what it assumes about the true spatial distribution of mass change, and the gap between missions and what you are and are not willing to claim about what happened in it. This lab has given you the tools to address all three in your own work.

The mass-loss numbers from this chapter will reappear in {doc}`../ice_flow/mass-balance` as observational targets. The models there are asked to reproduce not just the sign and magnitude of loss but its acceleration and its regional pattern. The fact that three independent measurement systems agree on those features is what gives the comparison between observation and model its power.

---

**Further reading.** The original GRACE mission overview is {cite}`tapley2004`. The first GRACE-based acceleration of Greenland mass loss is {cite}`velicogna2006`. For the IMBIE reconciliation, see [TODO-CITE: IMBIE team Antarctica 2018] and [TODO-CITE: IMBIE team Greenland 2020]. For an accessible treatment of mascon processing choices and their impact, see [TODO-CITE: Watkins mascon processing JPL 2015].